In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_directory = 'rca100/cases/t076'
task_file_path = f'{task_directory}/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

task_data

```
Out[1]:
```
The alert is titled "checkout 错误次数告警" (checkout error count alert) and was triggered on April 23, 2026, at 01:04:26 (UTC+8). The alert time window spans from 01:02:42.620252 to 01:06:26 (UTC+8). The affected entity is the operation `checkout::/oteldemo.CheckoutService/PlaceOrder` within the APM domain. The prompt requests an analysis of the root cause of this issue.

The original code execution output of IPython Kernel is also provided below for reference:

{'task_id': 't076', 'task_version': '1.0', 'alert_event_id': 'v827a017mt7fn0i5v9uphtkka5', 'alert_title': 'checkout 错误次数告警', 'alert_trigger_time': '2026-04-23T01:04:26+0800', 'alert_window': {'start': '2026-04-23T01:02:42.620252+08:00', 'end': '2026-04-23T01:06:26+08:00'}, 'alert_entity': {'entity_id': None, 'entity_name': 'checkout::/oteldemo.CheckoutService/PlaceOrder', 'entity_type': None, 'entity_domain': None}, 'prompt_text': '<code vibeops_object type="alert_event">checkout 错误次数告警<alert_event event_id="fcb16e5fcec8be3b09f808f02adf489d" trans_id="v827a017mt7fn0i5v9uphtkka5" rule_id="08493be2-a181-4480-9895-2b2c5607122e" rule_name="checkout 错误次数告警" entity_id="d219413245b68b297976412bbee076cf" entity_name="checkout::/oteldemo.CheckoutService/PlaceOrder" entity_type="apm.operation" entity_domain="apm" alert_time="2026-04-23T01:04:26+0800" /></code> 帮我分析下根因。', 'workspace': 'rca-benchmark', 'region_id': 'cn-hongkong', 'available_modalities': ['metrics', 'logs', 'traces', 'events', 'alerts', 'topology'], 'scoring_note': 'Output contract (prediction_schema.json) and fault taxonomy (taxonomy.json) will be published in a follow-up release.'}```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = f'{task_directory}/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# List all available KPIs and their associated components
kpi_components = metrics_data[['metric', 'entity_name']].drop_duplicates().sort_values(by=['metric', 'entity_name'])
kpi_components

```
Out[2]:
```
The `metrics.parquet` file contains 308 unique combinations of KPIs and their associated components. These include various metrics such as `apiserver_concurrent_connections`, `apiserver_requests_per_second`, and `workload`, associated with components like `inventory`, `payment`, `product-catalog`, `quote`, and `shipping`. To analyze anomalies, we can focus on KPIs relevant to the affected entity (`checkout::/oteldemo.CheckoutService/PlaceOrder`) and its dependencies.

The original code execution output of IPython Kernel is also provided below for reference:

metric      entity_name
51412     apiserver_concurrent_connections                 
51808   apiserver_non_2xx_read_request_qps                 
52072  apiserver_non_2xx_write_request_qps                 
52204        apiserver_requests_per_second                 
72607            arms_grouped_thread_count        inventory
...                                    ...              ...
72986                             workload        inventory
72993                             workload          payment
72997                             workload  product-catalog
73001                             workload            quote
73008                             workload         shipping

[308 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

# Ensure 'time' and 'value' columns are numeric for calculations
metrics_data['time'] = pd.to_numeric(metrics_data['time'], errors='coerce')
metrics_data['value'] = pd.to_numeric(metrics_data['value'], errors='coerce')

# Create a new column to classify by 'component-KPI'
metrics_data['component_kpi'] = metrics_data['entity_name'] + '-' + metrics_data['metric']

# Calculate the global P95 threshold for each 'component-KPI' time series
p95_thresholds = metrics_data.groupby('component_kpi')['value'].quantile(0.95).reset_index()
p95_thresholds.rename(columns={'value': 'p95_threshold'}, inplace=True)

p95_thresholds

```
Out[3]:
```
The global P95 thresholds for each 'component-KPI' time series have been calculated using the entire dataset. These thresholds represent the 95th percentile of metric values for each combination of component and KPI. For example, the P95 threshold for `apiserver_requests_per_second` is approximately 26.24, while for `shipping::oteldemo.ShippingService/GetQuote-response_time`, it is 6461.8. These thresholds can be used to identify anomalies by comparing actual values against them.

The original code execution output of IPython Kernel is also provided below for reference:

component_kpi  p95_threshold
0                    -apiserver_concurrent_connections       5.000000
1                  -apiserver_non_2xx_read_request_qps       0.000000
2                 -apiserver_non_2xx_write_request_qps       0.354510
3                       -apiserver_requests_per_second      26.243044
4                           -cluster_failed_node_count       0.000000
..                                                 ...            ...
303  shipping::oteldemo.ShippingService/GetQuote-av...       0.019491
304  shipping::oteldemo.ShippingService/GetQuote-re...    6461.800000
305  shipping::oteldemo.ShippingService/GetQuote-sl...      36.800000
306  shipping::oteldemo.ShippingService/ShipOrder-a...       0.000011
307  shipping::oteldemo.ShippingService/ShipOrder-r...    6244.250000

[308 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

from datetime import datetime
import pytz

# Define the alert time window in UTC+8
alert_start = datetime.fromisoformat('2026-04-23T01:02:42.620252+08:00').astimezone(pytz.timezone('Asia/Shanghai'))
alert_end = datetime.fromisoformat('2026-04-23T01:06:26+08:00').astimezone(pytz.timezone('Asia/Shanghai'))

# Convert the alert time window to microseconds for filtering
alert_start_us = int(alert_start.timestamp() * 1_000_000)
alert_end_us = int(alert_end.timestamp() * 1_000_000)

# Filter metrics data to include only the alert time window
filtered_metrics = metrics_data[(metrics_data['time'] >= alert_start_us) & (metrics_data['time'] <= alert_end_us)]

# Merge the filtered metrics with the P95 thresholds
merged_data = filtered_metrics.merge(p95_thresholds, on='component_kpi', how='left')

# Identify anomalies where the actual value exceeds the P95 threshold
anomalies = merged_data[merged_data['value'] > merged_data['p95_threshold']]

# Focus on the affected entity and its dependencies
affected_entity = 'checkout::/oteldemo.CheckoutService/PlaceOrder'
anomalies_filtered = anomalies[anomalies['entity_name'].str.contains(affected_entity)]

anomalies_filtered

```
Out[4]:
```
No anomalies were detected for the affected entity (`checkout::/oteldemo.CheckoutService/PlaceOrder`) or its dependencies during the alert time window when comparing actual metric values against their respective global P95 thresholds. This suggests that the issue may not be directly related to metric anomalies for this entity or its immediate dependencies. Further investigation into other telemetry data (e.g., logs or traces) may be required.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [time, domain, entity_set, entity_id, entity_name, metric, value, metric_set_id, service, component_kpi, p95_threshold]
Index: []

[0 rows x 11 columns]```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_file_path = f'{task_directory}/traces.parquet'
traces_data = pd.read_parquet(traces_file_path)

# Ensure relevant columns are numeric for filtering
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')
traces_data['duration'] = pd.to_numeric(traces_data['duration'], errors='coerce')

# Filter traces for the affected operation within the alert time window
affected_operation = 'checkout::/oteldemo.CheckoutService/PlaceOrder'
filtered_traces = traces_data[
    (traces_data['spanName'].str.contains(affected_operation)) &
    (traces_data['startTime'] >= alert_start_us * 1_000) &  # Convert to nanoseconds
    (traces_data['endTime'] <= alert_end_us * 1_000)        # Convert to nanoseconds
]

# Identify traces with errors or unusually high durations
error_traces = filtered_traces[filtered_traces['statusCode'] != 0]
slow_traces = filtered_traces.sort_values(by='duration', ascending=False).head(10)  # Top 10 slowest traces

error_traces, slow_traces

```
Out[5]:
```
No traces with errors or unusually high durations were found for the affected operation (`checkout::/oteldemo.CheckoutService/PlaceOrder`) within the alert time window. This indicates that the issue is not evident in the trace data for this operation during the specified period. Further investigation into other telemetry data sources may be necessary to identify the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [traceId, spanId, parentSpanId, kind, spanName, startTime, endTime, duration, serviceName, pid, hostname, statusCode, statusMessage, traceState, resources, attributes, events, __topic__, __tag__:__pack_id__, links, ip]
Index: []

[0 rows x 21 columns], Empty DataFrame
Columns: [traceId, spanId, parentSpanId, kind, spanName, startTime, endTime, duration, serviceName, pid, hostname, statusCode, statusMessage, traceState, resources, attributes, events, __topic__, __tag__:__pack_id__, links, ip]
Index: []

[0 rows x 21 columns])```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_file_path = f'{task_directory}/logs.parquet'
logs_data = pd.read_parquet(logs_file_path)

# Ensure '_time_' column is in a comparable format
logs_data['_time_'] = pd.to_numeric(logs_data['_time_'], errors='coerce')

# Filter logs for the affected operation within the alert time window
filtered_logs = logs_data[
    (logs_data['_time_'] >= alert_start_us) &
    (logs_data['_time_'] <= alert_end_us) &
    (logs_data['content'].str.contains(affected_operation, case=False, na=False))
]

# Search for error messages or anomalies in the log content
error_logs = filtered_logs[filtered_logs['content'].str.contains('error', case=False, na=False)]

error_logs

```
Out[6]:
```
No error messages or anomalies were found in the logs for the affected operation (`checkout::/oteldemo.CheckoutService/PlaceOrder`) within the alert time window. This suggests that the issue is not reflected in the log data for this operation during the specified period. Further analysis of other telemetry data sources may be required to identify the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _container_name_, _pod_name_, _namespace_, _pod_uid_, _container_ip_, _image_name_, __topic__, __tag__:__pack_id__, __tag__:_cluster_id_, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_]
Index: []

[0 rows x 15 columns]```
```

In [ ]:
# In[7]:

# Load the events.parquet file
events_file_path = f'{task_directory}/events.parquet'
events_data = pd.read_parquet(events_file_path)

# Ensure the event time column is in a comparable format
events_data['event_time'] = pd.to_numeric(events_data['eventId'].apply(lambda x: json.loads(x).get('lastTimestamp', 0)), errors='coerce')

# Filter events within the alert time window
filtered_events = events_data[
    (events_data['event_time'] >= alert_start_us) &
    (events_data['event_time'] <= alert_end_us)
]

# Look for events related to pod or node issues
pod_node_issues = filtered_events[
    filtered_events['eventId'].apply(lambda x: 'pod' in json.loads(x).get('involvedObject', {}).get('kind', '').lower() or
                                      'node' in json.loads(x).get('involvedObject', {}).get('kind', '').lower())
]

pod_node_issues

```
Out[7]:
```
No events related to pod or node issues, such as scheduling failures, crashes, or resource pressure, were found within the alert time window. This indicates that the issue is not reflected in the Kubernetes events data during the specified period. Further investigation into other telemetry data sources may be necessary to identify the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: []
Index: []```
```

In [ ]:
# In[8]:

# Load the topology.json file
topology_file_path = f'{task_directory}/topology.json'
with open(topology_file_path, 'r') as file:
    topology_data = json.load(file)

# Extract entities and edges from the topology data
entities = pd.DataFrame(topology_data['entities'])
edges = pd.DataFrame(topology_data['edges'])

# Focus on the affected operation and its relationships
affected_operation_id = entities[entities['name'] == 'checkout::/oteldemo.CheckoutService/PlaceOrder']['id'].values[0]

# Identify upstream and downstream services related to the affected operation
upstream_services = edges[(edges['dst'] == affected_operation_id) & (edges['relation'] == 'calls')]
downstream_services = edges[(edges['src'] == affected_operation_id) & (edges['relation'] == 'calls')]

upstream_services, downstream_services

```
Out[8]:
```
The service dependency graph analysis did not reveal any upstream or downstream services related to the affected operation (`checkout::/oteldemo.CheckoutService/PlaceOrder`). This suggests that there are no direct call relationships involving this operation in the topology data. Further investigation into other telemetry data sources or indirect causes may be necessary to identify the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [src, src_type, dst, dst_type, relation, first_observed, last_observed]
Index: [], Empty DataFrame
Columns: [src, src_type, dst, dst_type, relation, first_observed, last_observed]
Index: [])```
```